# *Fast Hands-on QLoRA (Quantized LoRA) – Colab GPU*

## **Installation**

In [1]:
# # === CLEAN INSTALL FOR COLAB GPU ===
# !pip install -q "bitsandbytes==0.46.1" "accelerate==0.30.0" "transformers==4.41.0" "peft==0.11.0" "datasets==2.20.0"
# !pip install -q langchain langchain-community

# print("✅ Packages installed")

### ***--------------RESTART SESSION---------------------***

## **Import the libraries**

In [2]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType

print("✅ Imports successful")

C:\Users\ashwi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



✅ Imports successful


## **Load Dataset**

In [3]:
# Small dataset for fast training
dataset = load_dataset("imdb", split="train[:600]")

def preprocess(example):
    return {"text": f"Review: {example['text'][:280]}\nSentiment:"}

dataset = dataset.map(preprocess)
print(f"Dataset size: {len(dataset)}")

Dataset size: 600


## **Load Model in 4-bit (QLoRA)**

In [4]:
# 4-bit Quantization Config
model_name = "gpt2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("✅ QLoRA 4-bit model loaded successfully!")

RuntimeError: No GPU found. A GPU is needed for quantization.

## **Apply LORA**

In [ ]:
# LoRA Configuration
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["c_attn"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 294,912 || all params: 124,734,720 || trainable%: 0.2364


## **Tokenization**

In [ ]:
# Tokenization + Training
def tokenize(example):
    tokens = tokenizer(example["text"], truncation=True, padding="max_length", max_length=128)
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = dataset.map(tokenize, batched=True, remove_columns=dataset.column_names)


Map:   0%|          | 0/600 [00:00<?, ? examples/s]

## **Training Stage ( runs < 2 mins )**

In [ ]:
# Training again with better settings
training_args = TrainingArguments(
    output_dir="./qlora_results",
    per_device_train_batch_size=4,
    num_train_epochs=1,
    logging_steps=20,
    save_strategy="epoch",           # Save at end of epoch
    save_total_limit=1,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

print("🚀 Training again...")
trainer.train()

# Force save the adapter
model.save_pretrained("./qlora_results/final_adapter")
tokenizer.save_pretrained("./qlora_results/final_adapter")

print("✅ Model saved to ./qlora_results/final_adapter")

🚀 Training again...


Step,Training Loss
20,4.233800
40,3.717700
60,3.269400
80,2.950500
100,2.790700
120,2.654700
140,2.623200


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


✅ Model saved to ./qlora_results/final_adapter


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [ ]:
import os
print("Files in final_adapter:")
print(os.listdir("./qlora_results/final_adapter"))

Files in final_adapter:
['merges.txt', 'tokenizer_config.json', 'vocab.json', 'special_tokens_map.json', 'README.md', 'tokenizer.json', 'adapter_config.json', 'adapter_model.safetensors']


## **Load the Fine-tuned Model for Inference**

In [ ]:
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

model = PeftModel.from_pretrained(
    base_model,
    "./qlora_results/final_adapter",   # ← Use this path
    device_map="auto"
)

model.eval()
print("✅ Adapter loaded successfully!")

✅ Adapter loaded successfully!


## **Testing Stage**

In [ ]:
def test_sentiment_fewshot(review):
    prompt = """Review: This was the best movie ever!
Sentiment: Positive

Review: I hated this film. Waste of time.
Sentiment: Negative

Review: """ + review + """
Sentiment:"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=15,
        do_sample=True,
        temperature=0.6,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id
    )

    return tokenizer.decode(output[0], skip_special_tokens=True)

print(test_sentiment_fewshot("This movie was absolutely fantastic!"))

Review: This was the best movie ever!
Sentiment: Positive

Review: I hated this film. Waste of time.
Sentiment: Negative

Review: This movie was absolutely fantastic!
Sentiment: Negative
